In [1]:
import sqlite3
import pandas as pd
import joblib
import re

def preprocess_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

def run_production():
   # 1. Load Live Records - add .copy()
    conn = sqlite3.connect('Classification.db')
    df = pd.read_sql_query("SELECT * FROM fake_news_classification", conn)
    live_df = df.iloc[55031+15723:].copy()
    conn.close()

    # 2. Load Assets
    model = joblib.load('fake_news_model.pkl')
    tfidf = joblib.load('tfidf_vectorizer.pkl')

    # 3. Preprocess & Predict
    live_df['clean_text'] = live_df['text'].apply(preprocess_text)
    X_live = tfidf.transform(live_df['clean_text'])
    
    live_df['prediction'] = model.predict(X_live)

    # 4. Show Sample Results
    print("Production Samples (Top 5):")
    print(live_df[['text', 'prediction']].head())
    
    # Save results to CSV for review
    live_df.to_csv('production_results.csv', index=False)

if __name__ == "__main__":
    run_production()


Production Samples (Top 5):
                                                    text  prediction
70754  Hillary Clinton Waiting In Wings Of Stage Sinc...           0
70755  UNITED NATIONS  —   Defying extraordinary pres...           1
70756  This is an old video that exposes so much abou...           0
70757  Ukrainian soldiers caught deserting are being ...           0
70758  Support for New Zealand s opposition Labour Pa...           1
